In [1]:
from data import api
import polars as pl

In [ ]:
# get panel data
df = api.get_panel_data()
df


In [ ]:
from models.GNN.gnn import (
    prepare_panel_for_gnn,
    fit_gnn_panel,
    rolling_cv_gnn,
    gnn_diagnostics_full,
)

# 1) Get your panel (same df as CAR / SARIMAX)
# df = api.get_panel_data()   # Polars DataFrame
outcome = "NUMBER_OF_HOMICIDIO"

X_stdzd, Y, edge_index, df_pd, meta_gnn = prepare_panel_for_gnn(
    df_polars=df,
    outcome=outcome,
    sector_cols=[f"S0{i}" for i in range(1, 9)],
    use_pca=True,
    pca_var=0.95,
    max_pcs=None,
    k_neighbors=4,
)

# 2) Fit NB-GNN on full panel
model_gnn, metrics_in, resid_gnn, mu_gnn = fit_gnn_panel(
    X_stdzd,
    Y,
    edge_index,
    gcn_hidden=32,
    gru_hidden=32,
    gru_layers=1,
    n_epochs=200,
)

print("GNN in-sample metrics:", metrics_in)

# 3) Diagnostics plots
resid_diag, mu_diag, diag_out = gnn_diagnostics_full(
    model_gnn,
    X_stdzd,
    Y,
    time_vals=meta_gnn["time_vals"],
    node_index=meta_gnn["dept_codes"],
)

# 4) Spatial residual maps (reuse your existing helpers)
# from your previous code:
# gdf_resid = build_residual_geodf(df_pd, resid_diag, "data/colombia_departments.geojson")
# plot_spatial_residuals(gdf_resid, value_col="resid", title="NB-GNN residuals")

# 5) Rolling CV
cv_gnn = rolling_cv_gnn(
    X_stdzd,
    Y,
    edge_index,
    time_vals=meta_gnn["time_vals"],
    n_folds=3,
    horizon_months=6,
)
print(cv_gnn)
